# Final Individual Project — The Shifting Landscape of Global Clinical Trials
## What 13,748 Pharmaceutical Trials (1984–2020) Reveal About Drug Development

**Author:** Shirisha Shashidhar Reddy
**Course:** Data Visualization — Summer 2026

**Dataset:** [ClinicalTrials.gov](https://clinicaltrials.gov) trial records for 10 major pharmaceutical sponsors, sourced via Kaggle ([*A Quick Overview of Clinical Trials*](https://www.kaggle.com/datasets/thedevastator/a-quick-overview-of-clinical-trials)) — real regulatory registry data, not a synthetic or teaching dataset.

**Why this dataset:** I spent three years in enterprise business analysis at IQVIA, working across major pharmaceutical accounts (AbbVie, Organon, Merck, Pfizer) on data governance, transparency reporting, and CRM systems that support clinical operations. This project lets me apply what I learned this semester to an industry I already understand from the inside — and ask questions I genuinely wanted answered.

**Links:**
- GitHub repository: *(added after publishing)*
- Live dashboard: *(added after deployment)*


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('data/clinical_trials.csv')
print(f"Loaded: {len(df):,} trials, {df.shape[1]} columns")
df.head()


---
## Preliminary Exploratory Data Analysis (EDA)


In [ ]:
print(df.info())
print()
print(df.describe(include='all').T)


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print(f"Year range: {df['Start_Year'].min()}–{df['Start_Year'].max()}")
print(f"Sponsors ({df['Sponsor'].nunique()}): {sorted(df['Sponsor'].unique())}")
print()
print("Phase counts:")
print(df['Phase'].value_counts(dropna=False))
print()
print(f"Unique conditions studied: {df['Condition'].nunique()}")
print()
print("Trial status counts:")
print(df['Status'].value_counts())


### Data quality & scope notes

A few honest caveats before diving into the analysis, rather than letting them surface as surprises later:

- **No spatial column.** This dataset does not include trial site country/location, so no map-based visualization is included. The brief encourages combining attribute types where possible, not requiring all four — this dataset still offers rich **categorical** (sponsor, phase, status, condition), **numerical** (enrollment), **temporal** (start year/month), and **text** (title, summary) dimensions.
- **417 trials (3%) have `Enrollment = 0`** — nearly all of these are `Withdrawn` trials that never actually enrolled a patient. These are excluded from any enrollment-size analysis (they would misleadingly cluster at zero) but are kept for status/outcome analyses.
- **263 trials (1.9%) are missing `Phase`**, and 144 are missing `Title`. These are excluded only from the specific analyses that use those fields.
- **2019–2020 appear to have very few trials** (49 and 2 respectively, versus 500+ in typical recent years) — this almost certainly reflects the dataset's collection cutoff rather than a real collapse in trial activity, and is called out explicitly where relevant below.


---
## Q1 — How does enrollment size differ across trial phases?

Trial phases progress from small safety studies (Phase 1) to large confirmatory trials (Phase 3) to post-market studies (Phase 4). Does enrollment scale the way the textbook says it should?


In [ ]:
main_phases = ['Phase 1', 'Phase 2', 'Phase 3', 'Phase 4']
q1 = df[(df['Enrollment'] > 0) & (df['Phase'].isin(main_phases))].copy()

fig1 = px.box(
    q1, x='Phase', y='Enrollment', category_orders={'Phase': main_phases},
    color_discrete_sequence=['#2E75B6'],  # single hue — one metric across ordered categories
    log_y=True,  # enrollment is heavily right-skewed (max 84K, median 124) — log scale needed
    labels={'Enrollment': 'Enrollment (log scale)', 'Phase': ''},
    title='Phase 3 trials enroll far more patients than any other phase — over 10x Phase 1, 2x Phase 4'
)
fig1.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                   xaxis=dict(showgrid=False), yaxis=dict(gridcolor='#EEEEEE'))
fig1.add_annotation(text='Trials with Enrollment = 0 excluded (never actually enrolled)',
                    xref='paper', yref='paper', x=1, y=-0.15, showarrow=False,
                    font=dict(size=10, color='#888888'))
fig1.show()


---
## Q2 — Which sponsor's trials are terminated or withdrawn most often, and is that linked to how many trials they run?

A high termination/withdrawal rate could signal riskier R&D bets — or just be noise from a sponsor running very few trials. This checks both at once.


In [ ]:
df['bad_outcome'] = df['Status'].isin(['Terminated', 'Withdrawn'])
sponsor_stats = df.groupby('Sponsor').agg(total=('NCT', 'count'), bad=('bad_outcome', 'sum')).reset_index()
sponsor_stats['rate'] = (sponsor_stats['bad'] / sponsor_stats['total'] * 100).round(1)
sponsor_stats = sponsor_stats.sort_values('rate', ascending=True)

# BBD HIGHLIGHT colour: orange marks the sponsor with the highest rate, blue-grey for the rest
sponsor_stats['highlight'] = sponsor_stats['Sponsor'].apply(
    lambda s: 'Highest' if s == sponsor_stats.loc[sponsor_stats['rate'].idxmax(), 'Sponsor'] else 'Other')

fig2 = px.bar(
    sponsor_stats, x='rate', y='Sponsor', orientation='h', color='highlight',
    color_discrete_map={'Highest': '#E07B39', 'Other': '#2E75B6'},  # CVD-safe blue/orange, no red-green
    labels={'rate': 'Terminated or Withdrawn (%)', 'Sponsor': ''},
    title="Pfizer's trials are terminated or withdrawn nearly 3x as often as AbbVie's — 17.5% vs 5.8%"
)
fig2.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                   showlegend=False, xaxis=dict(gridcolor='#EEEEEE', range=[0, sponsor_stats['rate'].max()*1.15]))
fig2.update_traces(marker_line_width=0)
fig2.show()

print(sponsor_stats[['Sponsor', 'total', 'bad', 'rate']].sort_values('rate', ascending=False))


---
## Q3 — Has the mix of trial phases shifted over time — is the industry moving toward earlier or later-stage research?

Comparing the 1990s, 2000s, and 2010s only (the 1980s and 2020s have too few trials — 1 and 2 respectively — to be meaningful).


In [ ]:
df['decade'] = (df['Start_Year'] // 10 * 10).astype(str) + 's'
q3 = df[df['Phase'].isin(main_phases) & df['decade'].isin(['1990s', '2000s', '2010s'])]
q3_ct = (pd.crosstab(q3['decade'], q3['Phase'], normalize='index') * 100).round(1)[main_phases]
q3_long = q3_ct.reset_index().melt(id_vars='decade', var_name='Phase', value_name='Share')

# BBD CATEGORICAL colour: Plotly's built-in CVD-safe qualitative palette (Okabe-Ito based)
fig3 = px.bar(
    q3_long, x='decade', y='Share', color='Phase', barmode='stack',
    category_orders={'decade': ['1990s', '2000s', '2010s'], 'Phase': main_phases},
    color_discrete_sequence=px.colors.qualitative.Safe,
    labels={'Share': 'Share of Trials (%)', 'decade': ''},
    title="Phase 1's share of trials grew six-fold since the 1990s (4% → 25%), while Phase 3's share nearly halved"
)
fig3.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                   yaxis=dict(gridcolor='#EEEEEE', range=[0, 100]), xaxis=dict(showgrid=False),
                   legend=dict(orientation='h', y=1.1))
fig3.show()


---
## Q4 — Which disease areas saw the sharpest rise or fall in trial activity between the 2000s and 2010s?

Looking at the 12 most-studied conditions overall, comparing trial counts in each decade.


In [ ]:
top_conditions = df['Condition'].value_counts().head(12).index.tolist()
q4 = df[df['Condition'].isin(top_conditions) & df['decade'].isin(['2000s', '2010s'])]
q4_ct = pd.crosstab(q4['Condition'], q4['decade'])

fig4 = go.Figure()
for cond in q4_ct.index:
    v2000s, v2010s = q4_ct.loc[cond, '2000s'], q4_ct.loc[cond, '2010s']
    increased = v2010s > v2000s
    # CVD-safe: blue = grew, orange = shrank (no red/green)
    color = '#2E75B6' if increased else '#E07B39'
    fig4.add_trace(go.Scatter(
        x=['2000s', '2010s'], y=[v2000s, v2010s], mode='lines+markers+text',
        line=dict(color=color, width=2.5), marker=dict(color=color, size=7),
        text=[f'{cond}: {v2000s}', f'{v2010s}'], textposition=['middle left', 'middle right'],
        textfont=dict(size=10), showlegend=False
    ))

fig4.update_layout(
    title='Hepatitis and COPD trial activity grew fastest since the 2000s — schizophrenia and hypertension fell sharpest',
    plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
    yaxis=dict(showticklabels=False, title='', showgrid=False),
    xaxis=dict(showgrid=False), margin=dict(l=180, r=140)
)
fig4.show()


---
## Q5 — Are early-phase trials more likely to be terminated or withdrawn than late-phase trials?

The intuition is that early, exploratory trials should be riskier. Is that actually true here?


In [ ]:
stage_map = {'Phase 1': 'Early (Phase 1/1-2)', 'Phase 1/Phase 2': 'Early (Phase 1/1-2)',
             'Phase 3': 'Late (Phase 3/4)', 'Phase 4': 'Late (Phase 3/4)'}
q5 = df[df['Phase'].isin(stage_map.keys())].copy()
q5['stage'] = q5['Phase'].map(stage_map)
q5_rate = q5.groupby('stage')['bad_outcome'].mean().mul(100).round(1).reset_index()
q5_rate.columns = ['stage', 'rate']

fig5 = px.bar(
    q5_rate, x='stage', y='rate',
    color_discrete_sequence=['#2E75B6'],  # single hue — comparing one metric across 2 groups
    labels={'rate': 'Terminated or Withdrawn (%)', 'stage': ''},
    title='Early- and late-phase trials fail at nearly the same rate (10.3% vs 9.9%) — risk is not concentrated in Phase 1'
)
fig5.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                   yaxis=dict(gridcolor='#EEEEEE', range=[0, max(q5_rate['rate'])*1.4]), xaxis=dict(showgrid=False))
fig5.update_traces(marker_line_width=0, texttemplate='%{y}%', textposition='outside')
fig5.show()


---
## Q6 — How do the 10 major sponsors differ in therapeutic focus?

Conditions are grouped into 7 broad therapeutic areas by keyword match (e.g. "Neoplasms/Carcinoma" → Oncology). Trials that don't clearly match any of the 7 areas are excluded from this specific chart, so the percentages reflect each sponsor's *categorized* portfolio, not their full trial count.


In [ ]:
def categorize(cond):
    if not isinstance(cond, str):
        return 'Other'
    c = cond.lower()
    if any(k in c for k in ['neoplasm','cancer','carcinoma','lymphoma','leukemia','melanoma','tumor','sarcoma','myeloma']):
        return 'Oncology'
    if any(k in c for k in ['diabetes','obesity','hypercholesterolemia','metabolic']):
        return 'Metabolic'
    if any(k in c for k in ['hepatitis','hiv','influenza','infection','tuberculosis','pneumonia']):
        return 'Infectious Disease'
    if any(k in c for k in ['asthma','pulmonary','copd','respiratory']):
        return 'Respiratory'
    if any(k in c for k in ['hypertension','cardiac','heart','coronary','atrial','cardiovascular']):
        return 'Cardiovascular'
    if any(k in c for k in ['alzheimer','parkinson','schizophrenia','depression','epilepsy','sclerosis','neuro']):
        return 'Neurological/Psychiatric'
    if any(k in c for k in ['arthritis','psoriasis','lupus','autoimmune']):
        return 'Autoimmune/Inflammatory'
    return 'Other'

df['category'] = df['Condition'].apply(categorize)
categorized = df[df['category'] != 'Other']

q6_ct = (pd.crosstab(categorized['Sponsor'], categorized['category'], normalize='index') * 100).round(1)
sponsor_order = q6_ct.max(axis=1).sort_values(ascending=False).index.tolist()

fig6 = px.imshow(
    q6_ct.loc[sponsor_order], text_auto=True, color_continuous_scale='Blues',  # sequential — a % share
    labels=dict(x='Therapeutic Area', y='Sponsor', color='Share (%)'),
    title='Gilead is the most specialized sponsor (62% infectious disease) — Roche and Bayer lead in oncology (49%+)'
)
fig6.update_layout(font=dict(family='Arial', size=11), margin=dict(t=60))
fig6.show()


---
## Q7 — Has typical trial size grown or shrunk over time?

Restricted to 1997–2018, where trial counts per year are large enough (35+) for a reliable median; earlier and later years are too sparse to trust.


In [ ]:
q7 = df[(df['Enrollment'] > 0) & (df['Start_Year'].between(1997, 2018))]
q7_med = q7.groupby('Start_Year')['Enrollment'].median().reset_index()

fig7 = px.line(
    q7_med, x='Start_Year', y='Enrollment',
    color_discrete_sequence=['#2E75B6'],
    labels={'Enrollment': 'Median Enrollment', 'Start_Year': ''},
    title='Median trial size has shrunk by nearly 40% since the late 1990s — from 220 patients in 1997 to 135 in 2018'
)
fig7.update_traces(line=dict(width=2.5))
fig7.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                   yaxis=dict(gridcolor='#EEEEEE', rangemode='tozero'), xaxis=dict(showgrid=False))
fig7.show()


---
## Q8 — Which sponsor completes the highest share of the trials it starts?

A simple proxy for R&D execution efficiency: `Completed` trials as a share of all trials run.


In [ ]:
df['completed'] = df['Status'] == 'Completed'
q8 = df.groupby('Sponsor')['completed'].mean().mul(100).round(1).reset_index()
q8.columns = ['Sponsor', 'completion_rate']
q8 = q8.sort_values('completion_rate')
q8['highlight'] = q8['Sponsor'].apply(
    lambda s: 'Lowest' if s == q8.loc[q8['completion_rate'].idxmin(), 'Sponsor'] else
              ('Highest' if s == q8.loc[q8['completion_rate'].idxmax(), 'Sponsor'] else 'Other'))

fig8 = px.bar(
    q8, x='completion_rate', y='Sponsor', orientation='h', color='highlight',
    color_discrete_map={'Highest': '#2E75B6', 'Lowest': '#E07B39', 'Other': '#DDDDDD'},
    labels={'completion_rate': 'Trials Completed (%)', 'Sponsor': ''},
    title='GSK completes 85.6% of the trials it starts — the highest of any major sponsor; AbbVie trails at 59.2%'
)
fig8.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                   showlegend=False, xaxis=dict(gridcolor='#EEEEEE', range=[0, 100]))
fig8.update_traces(marker_line_width=0)
fig8.show()


---
## Q9 — Among the most-studied conditions, which typically run the largest vs. smallest trials?


In [ ]:
q9 = df[(df['Enrollment'] > 0) & (df['Condition'].isin(top_conditions))]
q9_med = q9.groupby('Condition')['Enrollment'].median().sort_values().reset_index()

fig9 = px.bar(
    q9_med, x='Enrollment', y='Condition', orientation='h',
    color_discrete_sequence=['#2E75B6'],
    labels={'Enrollment': 'Median Enrollment', 'Condition': ''},
    title='Influenza trials enroll 3.4x as many patients as Alzheimer\'s trials among the most-studied conditions'
)
fig9.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                   xaxis=dict(gridcolor='#EEEEEE', range=[0, q9_med['Enrollment'].max()*1.15]),
                   yaxis=dict(showgrid=False))
fig9.update_traces(marker_line_width=0)
fig9.show()


---
## Q10 — Is there a seasonal pattern to when trials start?


In [ ]:
q10 = df['Start_Month'].value_counts().sort_index().reset_index()
q10.columns = ['month', 'count']
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
q10['month_name'] = q10['month'].apply(lambda m: month_names[m-1])
q10['highlight'] = q10['count'].apply(
    lambda c: 'Peak' if c == q10['count'].max() else ('Trough' if c == q10['count'].min() else 'Other'))

fig10 = px.bar(
    q10, x='month_name', y='count', color='highlight',
    category_orders={'month_name': month_names},
    color_discrete_map={'Peak': '#2E75B6', 'Trough': '#E07B39', 'Other': '#DDDDDD'},
    labels={'count': 'Trials Started', 'month_name': ''},
    title='Trial starts peak in October and dip in February — a modest but consistent seasonal rhythm'
)
fig10.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                    showlegend=False, yaxis=dict(gridcolor='#EEEEEE'), xaxis=dict(showgrid=False))
fig10.update_traces(marker_line_width=0)
fig10.show()


---
## Q11 — How has overall trial launch volume trended since the 1980s?


In [ ]:
q11 = df['Start_Year'].value_counts().sort_index().reset_index()
q11.columns = ['year', 'count']

fig11 = px.line(
    q11, x='year', y='count', color_discrete_sequence=['#2E75B6'],
    labels={'count': 'Trials Started', 'year': ''},
    title='Trial launches peaked around 2006–2007 and have declined by more than half since'
)
fig11.update_traces(line=dict(width=2.5))
fig11.add_vrect(x0=2019, x1=2020, fillcolor='#DDDDDD', opacity=0.4, line_width=0)
fig11.add_annotation(x=2019.5, y=q11['count'].max()*0.5, text='Likely incomplete —<br>dataset cutoff, not a<br>real slowdown',
                     showarrow=False, font=dict(size=10, color='#888888'))
fig11.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=12),
                    yaxis=dict(gridcolor='#EEEEEE', rangemode='tozero'), xaxis=dict(showgrid=False))
fig11.show()


---
## Conclusion — The Story

Three decades of pharmaceutical trial data tell a consistent story of an industry becoming **more cautious and more fragmented**:

1. **Trials are shrinking.** Median trial size fell nearly 40% between 1997 and 2018 (Q7), even as the industry runs more, smaller, earlier-stage studies — Phase 1's share of activity grew six-fold since the 1990s while Phase 3's share nearly halved (Q3). This points to a de-risking strategy: test more ideas cheaply before committing to expensive, large-scale trials.

2. **Termination risk isn't where you'd expect it.** Despite Phase 1 trials being framed as the riskiest, early- and late-phase trials fail (terminate/withdraw) at almost identical rates (Q5) — the real variation is by *sponsor*, not by phase: Pfizer's trials fail three times as often as AbbVie's (Q2), and completion rates range from 59% to 86% across the same ten companies (Q8).

3. **Sponsors are specializing, not generalizing.** Rather than spreading evenly across disease areas, major sponsors show clear therapeutic identities — Gilead concentrates 62% of its categorized trials in infectious disease, while Roche and Bayer lean heavily into oncology (Q6). Disease-area investment itself is shifting too: hepatitis and COPD trial activity grew fastest since the 2000s, while schizophrenia and hypertension trial activity fell sharply (Q4) — a pattern that roughly tracks where new drug classes (e.g. antivirals, biologics) opened up new possibilities.

Taken together: the clinical trials industry over the last 20 years looks less like a single pipeline and more like a portfolio of smaller, more specialized, more frequently-abandoned bets — a shift toward speed and optionality over scale.
